## `░ 1. Instalación de librerias`

In [ ]:
# %pip install sqlalchemy
# %pip install pandas
# %pip install oracledb 
%pip install openpyxl

#### `» 1.1 Importar librerias`

In [ ]:
import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.pool import NullPool
import oracledb

## `░ 2. Metodos globales y constantes`

#### `» 2.1 Constantes`

In [ ]:
# Conexión:

USER = 'devuser'
PASSWORD = 'D3vUs3r$2025'
HOST = '172.19.0.42'
PORT = '1521'
SID = 'INABIF02'

#### `» 2.1 Conexión a la base de datos Oracle`

In [ ]:
def get_engine():
   dsn = oracledb.makedsn(HOST, PORT, sid=SID)
   connection_string = f"oracle+oracledb://{USER}:{PASSWORD}@{dsn}"
   return create_engine(
      connection_string,
      poolclass=NullPool,  # ← AGREGADO: Sin pool para evitar problemas
      echo=False)  # Cambiar a True para debug)

#### `» 2.2 Métodos globales`

In [ ]:
# 2.2.1 Insertar DataFrame por lotes a la base de datos Oracle
def  insertar_dataframe_por_lotes(df, table_name, lote_size=1000):
   try:
         engine = get_engine()
         with engine.begin() as conn: # Maneja automáticamente commit/rollback
            df.to_sql(name=table_name, con=conn, if_exists='append', index=False, chunksize=lote_size)
            
         print(f'Datos insertados exitosamente en la tabla {table_name}.')
         return True, len(df)
   except Exception as e:
         print(f'Error al insertar datos: {e}')
         return False, 0
   finally:
         engine.dispose()
      

# 2.2.2 Ejecutar consulta y retornar DataFrame
def  execute_query(query, params=None):
   try:
         engine = get_engine()
         with engine.connect() as conn: # Cierra automáticamente la conexión
            if params:
               df = pd.read_sql_query(sql=text(query), con=conn, params=params)
            else:
               df = pd.read_sql_query(sql=text(query), con=conn)
            
         print(f'Consulta ejecutada exitosamente, filas encontradas: {len(df)}')
         return df
   except Exception as e:
         print(f"Error al ejecutar la consulta: {e}")
         return df.DataFrame()

## `░ 3. Exportar datos de oracle DB a archivos CSV`

#### `» 3.1 Constantes:`

In [ ]:
PATH_BASE = './data'

#### `» 3.2 Consulta sql para extraer datos de de Oracle DB`

In [ ]:
# Test
SQL_GET_TABLES = '''
            
         SELECT table_name
         FROM user_tables
         WHERE table_name LIKE 'SSI_' || '%'
         ORDER BY table_name

'''

SQL_GET_COLUMNS_INF = '''

   SELECT
      c.column_name AS Nombre_Campo,
      c.data_type AS Tipo_Dato,
      CASE
         WHEN c.data_type IN ('CHAR', 'VARCHAR2') THEN c.char_length
         ELSE c.data_precision
      END AS Longitud,
      CASE 
         WHEN pk.column_name IS NOT NULL THEN 'Sí'
         ELSE 'No'
      END AS Es_PK,
      CASE 
         WHEN fk.column_name IS NOT NULL THEN 'Sí'
         ELSE 'No'
      END AS Es_FK,
      fk.r_constraint_name AS Ref_Constraint
   FROM
      user_tab_cols c
   LEFT JOIN
      (
         SELECT a.column_name
         FROM all_cons_columns a
         JOIN all_constraints b ON a.constraint_name = b.constraint_name
         WHERE b.constraint_type = 'P' AND a.owner = USER AND a.table_name = :table_name
      ) pk ON c.column_name = pk.column_name
   LEFT JOIN
      (
         SELECT a.column_name, a.constraint_name AS r_constraint_name
         FROM all_cons_columns a
         JOIN all_constraints b ON a.constraint_name = b.constraint_name
         WHERE b.constraint_type = 'R' AND a.owner = USER AND a.table_name = :table_name
      ) fk ON c.column_name = fk.column_name
   WHERE
      c.table_name = :table_name
   ORDER BY c.column_id

'''

OUTPUT_FILE = "./data/ssi_diccionario_datos/diccionario_datos_familias_igualitarias.xlsx"


# 1. FUNCIÓN PRINCIPAL DE EXPORTACIÓN ---
def exportar_estructura_a_excel():

   # 1.1 Obtener la lista de tablas
   tablas = execute_query(SQL_GET_TABLES)
   
   # 1.2 Procesar cada tabla y exportarla a una hoja de Excel
   with pd.ExcelWriter(OUTPUT_FILE, engine='openpyxl') as writer:
      for i, row in tablas.iterrows():
         # Ejecutar la consulta de metadatos para la tabla actual
         df = execute_query(SQL_GET_COLUMNS_INF, params={'table_name': row.iloc[0]})
         
         # Exportar el DataFrame a una nueva hoja en el archivo Excel
         # Se limpia el nombre de la hoja para evitar errores de longitud/caracteres
         sheet_name = str(f'{row.iloc[0]}'[0:31])  # Limitar a 31 caracteres
         df.to_excel(writer, sheet_name=sheet_name, index=False)


exportar_estructura_a_excel()

In [ ]:
# df_result.to_excel(f'{PATH_BASE}/TGUNIDADORGANICA.xlsx', index=False)
rows = [row for row in tablas]

# 3.4. Procesar cada tabla y exportarla a una hoja de Excel
for i, row in tablas.iterrows():
   print(f"  -> Procesando tabla: {row.iloc[0]}")